# 00 — Setup and Configuration

Verifica el entorno, las dependencias, la configuración y la disponibilidad del corpus.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar configuración

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])


## 2. Verificar dependencias

In [ ]:

import importlib
packages = ["pandas", "numpy", "matplotlib", "networkx", "sklearn", "pycountry", "nbformat", "nbclient", "tabulate"]
rows = []
for package in packages:
    try:
        module = importlib.import_module(package)
        rows.append({"Package": package, "Status": "OK", "Version": getattr(module, "__version__", "installed")})
    except Exception as exc:
        rows.append({"Package": package, "Status": "MISSING", "Version": str(exc)})
status = pd.DataFrame(rows)
display(status)
save_csv(status, OUTPUT_DIR / "00_dependency_status.csv")
if (status["Status"] != "OK").any():
    raise RuntimeError("Instala las dependencias con: pip install -r requirements.txt")


## 3. Verificar corpus y metadatos de búsqueda

In [ ]:

df = load_scopus_csv(PROJECT_ROOT, CONFIG)
summary = pd.DataFrame([
    {"Metric": "Documents", "Value": len(df)},
    {"Metric": "Columns", "Value": len(df.columns)},
    {"Metric": "Database", "Value": CONFIG.get("search_database", "")},
    {"Metric": "Search date", "Value": CONFIG.get("search_date", "")},
    {"Metric": "Query label", "Value": CONFIG.get("query_label", "")},
    {"Metric": "Paper title", "Value": CONFIG.get("paper_title", "")},
])
display(summary)
save_csv(summary, OUTPUT_DIR / "00_project_summary.csv")
(OUTPUT_DIR / "00_search_query.txt").write_text(CONFIG.get("search_query", ""), encoding="utf-8")
